In [13]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_csv("powerplant_data.csv")

In [15]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
import torch
import torch.nn as nn

X_train_torch = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_torch = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_torch = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [19]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_torch, y_train_torch)
test_dataset = TensorDataset(X_test_torch, y_test_torch)

In [20]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [21]:

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
    
            
            nn.Linear(6, 6),
            nn.ReLU(),
    
            
            nn.Linear(6, 1),
        )

    def forward(self, x):
        return self.model(x)
    

In [22]:
import torch.optim as optim

model = ANN()

crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [23]:
# Train the ANN
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # tot training loss for 1 epoch
    
    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()
        
        outputs = model(xb) # forward prop....predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.. compute gradients
        optimizer.step() # params update
        
        running_loss += loss.item() # loss is a tensor => py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): # no gradients compute
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt") #.pt or .pth

epoch 1/100 ==> train loss = 206298.3462890625 & val loss = 204759.578125
epoch 2/100 ==> train loss = 196800.87174479166 & val loss = 182739.546875
epoch 3/100 ==> train loss = 157044.65924479166 & val loss = 127437.4296875
epoch 4/100 ==> train loss = 95718.36741536458 & val loss = 67548.8359375
epoch 5/100 ==> train loss = 48519.17766113281 & val loss = 35301.2578125
epoch 6/100 ==> train loss = 28193.17575276693 & val loss = 23451.0
epoch 7/100 ==> train loss = 20476.20380452474 & val loss = 18123.50390625
epoch 8/100 ==> train loss = 16235.331626383464 & val loss = 14389.5966796875
epoch 9/100 ==> train loss = 12797.204429117839 & val loss = 11109.998046875
epoch 10/100 ==> train loss = 9682.566383870442 & val loss = 8182.8154296875
epoch 11/100 ==> train loss = 7006.890995279948 & val loss = 5835.47802734375
epoch 12/100 ==> train loss = 4894.154509480794 & val loss = 4041.171142578125
epoch 13/100 ==> train loss = 3316.1538197835284 & val loss = 2757.321533203125
epoch 14/100 ==

In [24]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>